# shows 2022 2023

In [15]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

# =========================
# CONFIG
# =========================
INPUT_CSV = "sin_shows.csv"
N_TO_DOWNLOAD = 1

SLEEP_SECONDS = 1.1

REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"

if not REFRESH_TOKEN:
    raise ValueError("No encontré refresh token.")

BASE_DIR = Path("chartmetric_liveevents_2022_2023")
RAW_DIR = BASE_DIR / "raw_json_2022_2023"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "liveevents_2022_2023_summary.csv"
EVENTS_CSV = BASE_DIR / "liveevents_2022_2023_eventlevel.csv"

# Ventana: 2022-01-01 a 2023-12-31
# asumiendo hoy = 2026-04-04
FROM_DAYS_AGO = 1554
TO_DAYS_AGO = 825
LIMIT = 100

# =========================
# HELPERS
# =========================
def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return

    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

def read_input_artists(path: str):
    artists = []
    seen = set()

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_id = str(row.get("chartmetric_id", "")).strip()
            artist_name = str(row.get("artist_name", "")).strip()
            raw_rank = str(row.get("cm_artist_rank", "")).strip()

            if not raw_id:
                continue

            try:
                cmid = int(float(raw_id))
            except ValueError:
                continue

            try:
                cm_rank = int(float(raw_rank)) if raw_rank else 999999999
            except ValueError:
                cm_rank = 999999999

            if cmid in seen:
                continue
            seen.add(cmid)

            artists.append({
                "chartmetric_id": cmid,
                "artist_name": artist_name,
                "cm_artist_rank": cm_rank
            })

    artists.sort(key=lambda x: x["cm_artist_rank"])

    return artists

def read_existing_success_ids(summary_csv_path: Path):
    success_ids = set()

    if not summary_csv_path.exists():
        return success_ids

    with open(summary_csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ok_value = str(row.get("ok", "")).strip().lower()
            cmid = str(row.get("chartmetric_id", "")).strip()

            if ok_value == "true" and cmid:
                try:
                    success_ids.add(int(float(cmid)))
                except ValueError:
                    pass

    return success_ids

def parse_event_year(start_date_value):
    """
    start_date viene como string tipo '2023-12-05 21:00:00.000'
    """
    if not start_date_value:
        return None

    s = str(start_date_value).strip()
    if len(s) < 4:
        return None

    try:
        return int(s[:4])
    except ValueError:
        return None

def flatten_event(event: dict, artist: dict):
    city_obj = event.get("city") or {}
    event_year = parse_event_year(event.get("start_date"))

    return {
        "downloaded_at_utc": now_utc_iso(),
        "chartmetric_id": artist["chartmetric_id"],
        "artist_name": artist["artist_name"],
        "event_id": event.get("id"),
        "event_name": event.get("event_name"),
        "type": event.get("type"),
        "event_year": event_year,
        "start_date": event.get("start_date"),
        "end_date": event.get("end_date"),
        "venue_id": event.get("venue_id"),
        "venue_name": event.get("venue_name"),
        "venue_capacity": event.get("venue_capacity"),
        "city_id": city_obj.get("id"),
        "city_name": city_obj.get("name"),
        "code2": event.get("code2"),
        "price": event.get("price"),
        "low_price": event.get("low_price"),
        "high_price": event.get("high_price"),
        "currency": event.get("currency"),
        "is_headliner": event.get("is_headliner"),
        "jambase_event_url": event.get("jambase_event_url"),
        "songkick_event_url": event.get("songkick_event_url"),
        "seatgeek_event_url": event.get("seatgeek_event_url"),
        "ticketmaster_event_url": event.get("ticketmaster_event_url")
    }

# =========================
# READ INPUT + SKIP EXISTING
# =========================
artists = read_input_artists(INPUT_CSV)
already_downloaded_ids = read_existing_success_ids(SUMMARY_CSV)

pending_artists = [a for a in artists if a["chartmetric_id"] not in already_downloaded_ids]
batch_artists = pending_artists[:N_TO_DOWNLOAD]

print(f"Total artistas en input: {len(artists)}")
print(f"Ya descargados liveevents ok=True: {len(already_downloaded_ids)}")
print(f"Pendientes: {len(pending_artists)}")
print(f"Se descargarán ahora: {len(batch_artists)}")

if len(batch_artists) == 0:
    print("No hay artistas nuevos para descargar.")
    raise SystemExit

# =========================
# AUTH
# =========================
# Si existe access_token en memoria, lo reutilizamos.
# Si no existe, pedimos uno nuevo una sola vez.

if "access_token" in globals():
    print("\nReusando access_token ya obtenido en esta sesión.")
    if "headers" not in globals():
        headers = build_headers(access_token)
else:
    print("\nNo había access_token en memoria. Pidiendo uno nuevo...")
    access_token = get_access_token(REFRESH_TOKEN)
    headers = build_headers(access_token)
    print("Access token obtenido.")

summary_rows = []
event_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(batch_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    print(f"\n[{idx}/{len(batch_artists)}] {artist_name} ({artist_id})")

    all_events = []
    offset = 0
    n_requests_for_artist = 0
    last_rate_limit_limit = None
    last_rate_limit_remaining = None
    last_rate_limit_reset = None

    try:
        while True:
            time.sleep(SLEEP_SECONDS)

            url = f"https://api.chartmetric.com/api/artist/{artist_id}/past/events"
            params = {
                "fromDaysAgo": FROM_DAYS_AGO,
                "toDaysAgo": TO_DAYS_AGO,
                "limit": LIMIT,
                "offset": offset
            }

            resp = requests.get(url, headers=headers, params=params, timeout=90)

            if resp.status_code == 401:
                print("  401 -> renovando access token y reintentando...")
                access_token = get_access_token(REFRESH_TOKEN)
                headers = build_headers(access_token)
                time.sleep(SLEEP_SECONDS)
                resp = requests.get(url, headers=headers, params=params, timeout=90)

            n_requests_for_artist += 1

            last_rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
            last_rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
            last_rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

            if resp.status_code != 200:
                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": False,
                    "status_code": resp.status_code,
                    "error_text": resp.text[:500],
                    "raw_json_path": "",
                    "n_shows_2022": "",
                    "n_shows_2023": "",
                    "n_requests_live_2022_2023": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })
                print(f"  status={resp.status_code} error")
                break

            data = resp.json()
            page_events = data.get("obj", []) or []
            n_page = len(page_events)

            all_events.extend(page_events)

            print(
                f"  page offset={offset} | eventos={n_page} | "
                f"acumulados={len(all_events)} | remaining={last_rate_limit_remaining}"
            )

            if n_page < LIMIT:
                n_shows_2022 = 0
                n_shows_2023 = 0

                for ev in all_events:
                    year = parse_event_year(ev.get("start_date"))
                    if year == 2022:
                        n_shows_2022 += 1
                    elif year == 2023:
                        n_shows_2023 += 1

                raw_payload = {
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "fromDaysAgo": FROM_DAYS_AGO,
                    "toDaysAgo": TO_DAYS_AGO,
                    "limit": LIMIT,
                    "n_requests_live_2022_2023": n_requests_for_artist,
                    "n_shows_2022": n_shows_2022,
                    "n_shows_2023": n_shows_2023,
                    "obj": all_events
                }

                raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_liveevents_2022_2023.json"
                with open(raw_path, "w", encoding="utf-8") as f:
                    json.dump(raw_payload, f, ensure_ascii=False, indent=2)

                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": True,
                    "status_code": 200,
                    "error_text": "",
                    "raw_json_path": str(raw_path),
                    "n_shows_2022": n_shows_2022,
                    "n_shows_2023": n_shows_2023,
                    "n_requests_live_2022_2023": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })

                for ev in all_events:
                    event_rows.append(flatten_event(ev, artist))

                print(
                    f"  OK FINAL | n_shows_2022={n_shows_2022} | "
                    f"n_shows_2023={n_shows_2023} | "
                    f"requests={n_requests_for_artist}"
                )
                break

            offset += LIMIT

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": now_utc_iso(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "ok": False,
            "status_code": "",
            "error_text": repr(e),
            "raw_json_path": "",
            "n_shows_2022": "",
            "n_shows_2023": "",
            "n_requests_live_2022_2023": n_requests_for_artist,
            "x_ratelimit_limit": last_rate_limit_limit,
            "x_ratelimit_remaining": last_rate_limit_remaining,
            "x_ratelimit_reset": last_rate_limit_reset
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc",
    "chartmetric_id",
    "artist_name",
    "ok",
    "status_code",
    "error_text",
    "raw_json_path",
    "n_shows_2022",
    "n_shows_2023",
    "n_requests_live_2022_2023",
    "x_ratelimit_limit",
    "x_ratelimit_remaining",
    "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if event_rows:
    event_fields = list(event_rows[0].keys())
    write_csv_rows(EVENTS_CSV, event_rows, event_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Event-level: {EVENTS_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"Nuevos OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")

Total artistas en input: 1826
Ya descargados liveevents ok=True: 279
Pendientes: 1547
Se descargarán ahora: 1

Reusando access_token ya obtenido en esta sesión.

[1/1] Willy William (210819)
  401 -> renovando access token y reintentando...
  page offset=0 | eventos=0 | acumulados=0 | remaining=206
  OK FINAL | n_shows_2022=0 | n_shows_2023=0 | requests=1

Listo.
Resumen: chartmetric_liveevents_2022_2023\liveevents_2022_2023_summary.csv
Event-level: chartmetric_liveevents_2022_2023\liveevents_2022_2023_eventlevel.csv
Raw JSON dir: chartmetric_liveevents_2022_2023\raw_json_2022_2023
Nuevos OK: 1 / 1


In [14]:
import time
import requests

ARTIST_ID = 439
ARTIST_NAME = "Coldplay"



SLEEP_SECONDS = 1.1
LIMIT = 20

FROM_DAYS_AGO = 1554
TO_DAYS_AGO = 825

WINDOWS = [
    {"label": "2022_2023_completo", "fromDaysAgo": 1554, "toDaysAgo": 825},
    {"label": "solo_2023", "fromDaysAgo": 1189, "toDaysAgo": 825},
    {"label": "solo_2022", "fromDaysAgo": 1554, "toDaysAgo": 1190},
    {"label": "mitad_2023", "fromDaysAgo": 1000, "toDaysAgo": 900},
]


if "access_token" not in globals():
    access_token = get_access_token(REFRESH_TOKEN)

headers = build_headers(access_token)

print(f"Artista: {ARTIST_NAME} ({ARTIST_ID})")

for i, w in enumerate(WINDOWS, start=1):

    print("\n" + "=" * 70)
    print(f"[{i}/{len(WINDOWS)}] {w['label']}")

    url = f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/past/events"
    params = {
        "fromDaysAgo": w["fromDaysAgo"],
        "toDaysAgo": w["toDaysAgo"],
        "limit": LIMIT,
        "offset": 0
    }

    time.sleep(SLEEP_SECONDS)
    resp = requests.get(url, headers=headers, params=params, timeout=90)

    print("status_code:", resp.status_code)

    if resp.status_code != 200:
        print("error:", resp.text)
        continue

    data = resp.json()
    events = data.get("obj", []) or []

    print("eventos devueltos:", len(events))

    for ev in events[:5]:
        print(ev.get("start_date"), ev.get("event_name"))

Artista: Coldplay (439)

[1/4] 2022_2023_completo
status_code: 200
eventos devueltos: 0

[2/4] solo_2023
status_code: 200
eventos devueltos: 0

[3/4] solo_2022
status_code: 200
eventos devueltos: 0

[4/4] mitad_2023
status_code: 200
eventos devueltos: 0


## duplicados

In [ ]:
import csv
from pathlib import Path
from collections import defaultdict

INPUT_SUMMARY = Path("chartmetric_liveevents_2022_2023/liveevents_2022_2023_summary.csv")

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

if not INPUT_SUMMARY.exists():
    raise FileNotFoundError(f"No existe el archivo: {INPUT_SUMMARY}")

with open(INPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("El summary está vacío.")

rows_by_id = defaultdict(list)

for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        rows_by_id[cmid].append(row)

duplicates = {cmid: group for cmid, group in rows_by_id.items() if len(group) > 1}

total_ids = len(rows_by_id)
total_duplicates = sum(len(group) - 1 for group in duplicates.values())

print("=== DETECCIÓN DE DUPLICADOS ===")
print("Archivo:", INPUT_SUMMARY)
print("Filas totales:", len(rows))
print("IDs únicos:", total_ids)
print("Duplicados totales:", total_duplicates)
print("Artistas con duplicados:", len(duplicates))

if not duplicates:
    print("\nNo se detectaron duplicados.")
else:
    print("\nPrimeros 20 artistas con duplicados:")
    for cmid, group in list(duplicates.items())[:20]:
        print(f"{cmid} | repeticiones: {len(group)} | artist_name: {group[-1].get('artist_name', '')}")

## detalle de duplicados

In [ ]:
import csv
from collections import defaultdict

SUMMARY_FILE = "chartmetric_liveevents_2022_2023/liveevents_2022_2023_summary.csv"

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

with open(SUMMARY_FILE, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

rows_by_id = defaultdict(list)

for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        rows_by_id[cmid].append(row)

duplicates = {k: v for k, v in rows_by_id.items() if len(v) > 1}

print("=== DUPLICADOS DETECTADOS 2022_2023 ===")
print("Cantidad de artistas duplicados:", len(duplicates))
print()

for cmid, group in duplicates.items():
    print("======================================")
    print(f"chartmetric_id: {cmid}")
    print(f"repeticiones: {len(group)}")
    
    for i, row in enumerate(group, start=1):
        print(f"\n--- fila {i} ---")
        print("artist_name:", row.get("artist_name"))
        print("ok:", row.get("ok"))
        print("status_code:", row.get("status_code"))
        print("n_shows_2022:", row.get("n_shows_2022"))
        print("n_shows_2023:", row.get("n_shows_2023"))
        print("n_requests:", row.get("n_requests_live_2022_2023"))
        print("downloaded_at:", row.get("downloaded_at_utc"))
    
    print()

## limpiar duplicadosimport csv
from pathlib import Path

INPUT_SUMMARY = Path("chartmetric_liveevents_2022_2023/liveevents_2022_2023_summary.csv")
OUTPUT_SUMMARY = Path("chartmetric_liveevents_2022_2023/liveevents_2022_2023_summary_dedup.csv")

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def count_unique_ids(rows, id_col="chartmetric_id"):
    ids = []
    for row in rows:
        cmid = to_int(row.get(id_col, ""))
        if cmid:
            ids.append(cmid)
    return len(set(ids)), len(ids) - len(set(ids))

if not INPUT_SUMMARY.exists():
    raise FileNotFoundError(f"No existe el archivo de entrada: {INPUT_SUMMARY}")

with open(INPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("El summary está vacío.")

fieldnames = list(rows[0].keys())

latest_by_id = {}
for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        latest_by_id[cmid] = row

dedup_rows = list(latest_by_id.values())

if not dedup_rows:
    raise ValueError("La deduplicación produjo 0 filas. Proceso cancelado.")

with open(OUTPUT_SUMMARY, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(dedup_rows)

with open(OUTPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    check_rows = list(csv.DictReader(f))

if not check_rows:
    raise ValueError("El archivo deduplicado se escribió vacío. Proceso cancelado.")

unique_ids_before, dup_before = count_unique_ids(rows)
unique_ids_after, dup_after = count_unique_ids(check_rows)

print("=== LIMPIEZA SUMMARY ===")
print("Filas originales:", len(rows))
print("IDs únicos originales:", unique_ids_before)
print("Duplicados originales:", dup_before)
print("Filas deduplicadas:", len(check_rows))
print("IDs únicos deduplicados:", unique_ids_after)
print("Duplicados deduplicados:", dup_after)
print("Archivo deduplicado creado:", OUTPUT_SUMMARY)

if dup_after != 0:
    raise ValueError("El archivo deduplicado todavía tiene duplicados. Proceso cancelado.")

## reasignar filenames

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime
import csv

BASE_DIR = Path("chartmetric_liveevents_2022_2023")

original = BASE_DIR / "liveevents_2022_2023_summary.csv"
dedup = BASE_DIR / "liveevents_2022_2023_summary_dedup.csv"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = BASE_DIR / f"liveevents_2022_2023_summary_backup_{timestamp}.csv"

def count_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return len(rows)

if not dedup.exists():
    raise FileNotFoundError(f"No existe el archivo deduplicado: {dedup}")

dedup_rows = count_rows(dedup)
if dedup_rows == 0:
    raise ValueError("El archivo deduplicado está vacío. Reemplazo cancelado.")

if original.exists():
    shutil.copy2(original, backup)
    print("Backup creado:", backup.name)
else:
    print("No existe archivo original. El deduplicado pasará a ser el oficial sin backup previo.")

shutil.copy2(dedup, original)
print("Archivo deduplicado copiado como oficial:", original.name)

final_rows = count_rows(original)
if final_rows != dedup_rows:
    raise ValueError(
        f"Verificación fallida: original final={final_rows}, dedup={dedup_rows}. "
        "El backup quedó disponible para recuperación."
    )

print("=== REEMPLAZO COMPLETADO ===")
print("Filas en dedup:", dedup_rows)
print("Filas en original final:", final_rows)
print("Proceso terminado.")

# control 2022 2023

In [ ]:
import csv

# =========================
# CONFIG
# =========================
INPUT_FILE = "sin_shows.csv"
LIVE_SUMMARY_FILE = "chartmetric_liveevents_2022_2023/liveevents_2022_2023_summary.csv"

# =========================
# HELPERS
# =========================
def read_csv_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def to_bool_str(value):
    return str(value).strip().lower() == "true"

# =========================
# READ FILES
# =========================
input_rows = read_csv_rows(INPUT_FILE)
live_rows = read_csv_rows(LIVE_SUMMARY_FILE)

# =========================
# TOTAL INPUT
# =========================
input_ids = []
seen_input = set()

for row in input_rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid and cmid not in seen_input:
        seen_input.add(cmid)
        input_ids.append(cmid)

input_ids_set = set(input_ids)

# =========================
# LIVE EVENTS STATUS
# Si un artista aparece varias veces en el summary,
# nos quedamos con la última fila.
# =========================
live_by_id = {}

for row in live_rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        live_by_id[cmid] = row

live_ids_set = set(live_by_id.keys())

ok_ids = set()
error_ids = set()
pagination_ids = set()
pagination_ok_ids = set()
zero_show_ok_ids = set()
positive_show_ok_ids = set()

for cmid, row in live_by_id.items():
    ok = to_bool_str(row.get("ok", ""))
    n_requests = to_int(row.get("n_requests_live_2022_2023", "0"))
    n_2022 = to_int(row.get("n_shows_2022", "0"))
    n_2023 = to_int(row.get("n_shows_2023", "0"))
    total_shows = n_2022 + n_2023

    if ok:
        ok_ids.add(cmid)
        if total_shows == 0:
            zero_show_ok_ids.add(cmid)
        else:
            positive_show_ok_ids.add(cmid)
    else:
        error_ids.add(cmid)

    if n_requests > 1:
        pagination_ids.add(cmid)
        if ok:
            pagination_ok_ids.add(cmid)

pending_ids = input_ids_set - ok_ids
pending_because_error_ids = error_ids & pending_ids

# =========================
# PRINT MAIN STATUS
# =========================
print("=== STATUS GENERAL ===")
print("Total artistas en input:", len(input_ids_set))
print("Total filas live summary:", len(live_rows))
print("Artistas únicos vistos en live summary:", len(live_ids_set))
print("OK live events:", len(ok_ids))
print("Errores live events:", len(error_ids))
print("Pendientes live events:", len(pending_ids))

print("\n=== COBERTURA DE SHOWS ===")
print("OK con 0 shows (2022+2023):", len(zero_show_ok_ids))
print("OK con >0 shows (2022+2023):", len(positive_show_ok_ids))

print("\n=== PAGINACIÓN ===")
print("Artistas que requirieron más de 1 request:", len(pagination_ids))
print("De esos, resueltos OK:", len(pagination_ok_ids))
print("De esos, NO resueltos OK:", len(pagination_ids - pagination_ok_ids))

print("\n=== PENDIENTES (primeros 20) ===")
pending_preview = sorted(list(pending_ids))[:20]
print(pending_preview)

print("\n=== ERRORES / REINTENTAR (primeros 20) ===")
retry_preview = sorted(list(pending_because_error_ids))[:20]
print(retry_preview)

print("\n=== TOP 20 artistas por cantidad de requests live ===")
top_requests = []

for cmid, row in live_by_id.items():
    top_requests.append({
        "chartmetric_id": cmid,
        "artist_name": row.get("artist_name", ""),
        "ok": row.get("ok", ""),
        "n_requests_live_2022_2023": to_int(row.get("n_requests_live_2022_2023", "0")),
        "n_shows_2022": to_int(row.get("n_shows_2022", "0")),
        "n_shows_2023": to_int(row.get("n_shows_2023", "0")),
    })

top_requests = sorted(
    top_requests,
    key=lambda x: (
        x["n_requests_live_2022_2023"],
        x["n_shows_2022"] + x["n_shows_2023"]
    ),
    reverse=True
)

for row in top_requests[:20]:
    print(row)